# 04 · Controlled ROI comparison: 68 vs 18 regions (development CV only)

`configs/final/roi_comparison.yaml`. The same 52 development participants, the same five participant-disjoint
folds (`xu_dev_cv`), window mode B, the same training procedures and seeds, LCMV and wMNE, for:

- `exp1_source_psd_de`: `logreg__psd`, `svm__psd`, `logreg__de`, `svm__de`
- `exp4_band_mean`: `plv__psd`
- `exp5_multibranch`: `plv__separate__mean`

GNNs on dense weighted PLV, exactly 60 epochs. **The holdout is not evaluated** (and is refused by the code until
the final analysis is frozen in notebook 99).

The report keeps two kinds of evidence apart: **source localization / spatial resolution** (label-free: forced
redundancy, cross-talk, own-region dominance of the operators actually built, feature reliability; notebook 02) and
**classification** (participant-level pooled out-of-fold ROC-AUC with DeLong CI, balanced accuracy, F1, sensitivity,
specificity, paired r18 - dk68 differences with paired DeLong and participant-bootstrap CIs). A higher AUC alone
does not make an atlas better, and the summary never forces a winner. You then record the analysis atlas (§4).

The dk68 runs are the same units the experiment notebooks reuse (nothing is computed twice). GPU runtime (T4).
Resumable.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
import json

## 1. Both atlases' stores verify (read-only)

In [ ]:
for atlas in ('dk68', 'r18'):
    sh(f"python scripts/final.py stores --atlas {atlas}")

## 2. Runs (development CV only; each unit is checkpointed)

In [ ]:
RUNS = [["exp1", "logreg__psd", "dk68", "lcmv", "none"], ["exp1", "svm__psd", "dk68", "lcmv", "none"], ["exp1", "logreg__de", "dk68", "lcmv", "none"], ["exp1", "svm__de", "dk68", "lcmv", "none"], ["exp4", "plv__psd", "dk68", "lcmv", "dense"], ["exp5", "plv__separate__mean", "dk68", "lcmv", "dense"], ["exp1", "logreg__psd", "dk68", "wmne", "none"], ["exp1", "svm__psd", "dk68", "wmne", "none"], ["exp1", "logreg__de", "dk68", "wmne", "none"], ["exp1", "svm__de", "dk68", "wmne", "none"], ["exp4", "plv__psd", "dk68", "wmne", "dense"], ["exp5", "plv__separate__mean", "dk68", "wmne", "dense"], ["exp1", "logreg__psd", "r18", "lcmv", "none"], ["exp1", "svm__psd", "r18", "lcmv", "none"], ["exp1", "logreg__de", "r18", "lcmv", "none"], ["exp1", "svm__de", "r18", "lcmv", "none"], ["exp4", "plv__psd", "r18", "lcmv", "dense"], ["exp5", "plv__separate__mean", "r18", "lcmv", "dense"], ["exp1", "logreg__psd", "r18", "wmne", "none"], ["exp1", "svm__psd", "r18", "wmne", "none"], ["exp1", "logreg__de", "r18", "wmne", "none"], ["exp1", "svm__de", "r18", "wmne", "none"], ["exp4", "plv__psd", "r18", "wmne", "dense"], ["exp5", "plv__separate__mean", "r18", "wmne", "dense"]]      # (experiment, variant, atlas, inverse, graph)
ERRORS = []
for exp, variant, atlas, inverse, graph in RUNS:
    g = '' if graph == 'none' else f' --graph {graph}'
    try:
        sh(f"python scripts/final.py run --exp {exp} --inverse {inverse} --mode B --atlas {atlas}{g} "
           f"--variants {variant}")
    except RuntimeError:
        ERRORS.append((exp, variant, atlas, inverse))
if ERRORS:
    raise RuntimeError(f'units did not complete: {ERRORS}; completed units are saved. Run all again.')

## 3. The comparison (saved outputs only)

In [ ]:
try:
    sh("python scripts/final.py roi-compare")
finally:
    d = latest(FINAL + '/roi_comparison/compare_*')
    from IPython.display import Markdown
    if d:
        display(Markdown(open(d + '/report.md').read()))

## 4. Your decision: the analysis atlas (recorded once)

Review §3 (source-resolution evidence first, classification second). Then set `ATLAS_DECISION` to `'dk68'` or
`'r18'` with the scientific reason and run this cell. Nothing is recorded while it is `None`; notebook 03 and the
experiment notebooks stop until a decision exists. The atlas not chosen stays a sensitivity / reference analysis.

In [ ]:
ATLAS_DECISION = None   # 'dk68' or 'r18'
REASON = ''             # e.g. 'r18: no forced redundancy, cross-talk 0.5 vs 0.85, reliability ...; AUC difference ...'
if ATLAS_DECISION:
    import shlex
    sh(f"python scripts/final.py choose-atlas --atlas {ATLAS_DECISION} --reason {shlex.quote(REASON)}")
else:
    print('no atlas recorded yet: set ATLAS_DECISION and REASON above after reviewing section 3')